### from Pytorch and Tensorflow

```python
# 1. Imports

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import (
    TensorDataset,
    DataLoader
)

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# General
import numpy as np
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)



# 2. Train/Test Split + Scaling
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



# 3. PyTorch Neural Network
class RegressionNN(nn.Module): # nn.Module is the base class for all PyTorch neural networks

    def __init__(self, n_features):

        super().__init__() # initializes the parent class (nn.Module)

        self.network = nn.Sequential(

            nn.Linear(n_features, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)


model_pt = RegressionNN(
    n_features=X_train_scaled.shape[1]
)

model_pt




# 4. PyTorch Dataset + DataLoader
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values.reshape(-1, 1),
    dtype=torch.float32
)

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)



# 5. PyTorch Loss + Optimizer
criterion = nn.MSELoss()

optimizer = optim.Adam(
    model_pt.parameters(),
    lr=0.001
)



# 6. PyTorch Training Loop
epochs = 100

for epoch in range(epochs):

    model_pt.train()

    epoch_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        predictions = model_pt(X_batch)

        loss = criterion(
            predictions,
            y_batch
        )

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    if epoch % 10 == 0:

        print(
            f"epoch={epoch:4d}, "
            f"loss={epoch_loss:.6f}"
        )




# 7. PyTorch Evaluation
model_pt.eval()

with torch.no_grad():

    X_test_tensor = torch.tensor(
        X_test_scaled,
        dtype=torch.float32
    )

    y_pred_pt = (
        model_pt(X_test_tensor)
        .numpy()
        .flatten()
    )

mean_squared_error(
    y_test,
    y_pred_pt
)

mean_absolute_error(
    y_test,
    y_pred_pt
)

r2_score(
    y_test,
    y_pred_pt
)



# 8. PyTorch Model Parameters
for name, param in model_pt.named_parameters():

    print(
        name,
        param.shape
    )




# 9. TensorFlow / Keras Neural Network
model_tf = Sequential([

    Dense(
        128,
        activation="relu",
        input_shape=(
            X_train_scaled.shape[1],
        )
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(1)
])

model_tf.summary()




# 10. TensorFlow Compile
model_tf.compile(

    optimizer="adam",

    loss="mse",

    metrics=[
        "mae"
    ]
)




# 11. TensorFlow Training
history = model_tf.fit(

    X_train_scaled,
    y_train,

    validation_split=0.2,

    epochs=100,

    batch_size=32,

    verbose=1
)




# 12. TensorFlow Early Stopping
early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    restore_best_weights=True
)

model_tf.fit(

    X_train_scaled,
    y_train,

    validation_split=0.2,

    epochs=1000,

    callbacks=[
        early_stopping
    ],

    verbose=1
)




# 13. TensorFlow Evaluation

y_pred_tf = (
    model_tf.predict(
        X_test_scaled
    )
    .flatten()
)

mean_squared_error(
    y_test,
    y_pred_tf
)

mean_absolute_error(
    y_test,
    y_pred_tf
)

r2_score(
    y_test,
    y_pred_tf
)




# 14. TensorFlow Training History

import matplotlib.pyplot as plt

plt.plot(
    history.history["loss"],
    label="train"
)

plt.plot(
    history.history["val_loss"],
    label="validation"
)

plt.legend()

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.show()




# 15. Different Architectures

# Small Network
small_tf = Sequential([
    Dense(
        16,
        activation="relu"
    ),
    Dense(1)
])

# Medium Network
medium_tf = Sequential([
    Dense(
        64,
        activation="relu"
    ),
    Dense(
        32,
        activation="relu"
    ),
    Dense(1)
])

# Deep Network
deep_tf = Sequential([
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        64,
        activation="relu"
    ),
    Dense(
        32,
        activation="relu"
    ),
    Dense(1)
])




# 16. Common Loss Functions

# Mean Squared Error
nn.MSELoss()
tf.keras.losses.MeanSquaredError()

# Mean Absolute Error
nn.L1Loss()
tf.keras.losses.MeanAbsoluteError()

# Huber Loss
nn.HuberLoss()
tf.keras.losses.Huber()




# 17. Common Optimizers

# Adam
optim.Adam(
    model_pt.parameters(),
    lr=0.001
)

# SGD
optim.SGD(
    model_pt.parameters(),
    lr=0.01
)

# RMSprop
optim.RMSprop(
    model_pt.parameters(),
    lr=0.001
)

# TensorFlow
tf.keras.optimizers.Adam()
tf.keras.optimizers.SGD()
tf.keras.optimizers.RMSprop()




# 18. Save / Load Models

# PyTorch
torch.save(
    model_pt.state_dict(),
    "model.pt"
)

model_pt.load_state_dict(
    torch.load("model.pt")
)

# TensorFlow
model_tf.save(
    "model.keras"
)

loaded_model = (
    tf.keras.models.load_model(
        "model.keras"
    )
)




# 19. Prediction on New Data
X_new = X_test_scaled[:5]

# PyTorch
with torch.no_grad():

    pred_pt = model_pt(
        torch.tensor(
            X_new,
            dtype=torch.float32
        )
    )

# TensorFlow
pred_tf = model_tf.predict(
    X_new
)




# 20. Important Notes

# - Scale features before training
# - ReLU is the default activation to start with
# - Adam is usually the best first optimizer
# - MSE is the standard regression loss
# - Use validation data to detect overfitting
# - EarlyStopping is highly recommended
# - More layers = more capacity but more overfitting risk
# - PyTorch requires explicit training loops
# - TensorFlow/Keras automates most training steps
# - Monitor train/validation loss curves
# - Save trained models for deployment
```


### PyTorch + TensorFlow Classification (Binary / Multi-class)

```python


# Imports
import torch
import torch.nn as nn
import torch.optim as optim
import tensorflow as tf

from torch.utils.data import TensorDataset, DataLoader
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# Train/Test Split + Scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



# PyTorch

# Binary Classification Network
class BinaryNN(nn.Module):

    def __init__(self, n_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)


model_pt = BinaryNN(X_train_scaled.shape[1])

# Dataset
X_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

y_tensor = torch.tensor(
    y_train.values.reshape(-1, 1),
    dtype=torch.float32
)

loader = DataLoader(
    TensorDataset(X_tensor, y_tensor),
    batch_size=32,
    shuffle=True
)

# Loss + Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(
    model_pt.parameters(),
    lr=0.001
)

# Training
for epoch in range(100):

    for X_batch, y_batch in loader:

        optimizer.zero_grad()

        logits = model_pt(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()
        optimizer.step()


# Evaluation
model_pt.eval()

with torch.no_grad():

    logits = model_pt(
        torch.tensor(
            X_test_scaled,
            dtype=torch.float32
        )
    )

    probs = torch.sigmoid(logits)

    y_pred_pt = (
        probs >= 0.5
    ).int().numpy().flatten()

accuracy_score(
    y_test,
    y_pred_pt
)

print(
    classification_report(
        y_test,
        y_pred_pt
    )
)

confusion_matrix(
    y_test,
    y_pred_pt
)



# TensorFlow / Keras

# Binary Classification
model_tf = Sequential([
    Dense(
        64,
        activation="relu",
        input_shape=(
            X_train_scaled.shape[1],
        )
    ),
    Dense(
        1,
        activation="sigmoid"
    )
])

model_tf.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model_tf.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    verbose=1
)


# Early Stopping
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

model_tf.fit(
    X_train_scaled,
    y_train,
    epochs=1000,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)


# Evaluation
y_prob_tf = model_tf.predict(
    X_test_scaled
)

y_pred_tf = (
    y_prob_tf >= 0.5
).astype(int)

accuracy_score(
    y_test,
    y_pred_tf
)

print(
    classification_report(
        y_test,
        y_pred_tf
    )
)

confusion_matrix(
    y_test,
    y_pred_tf
)



# Multi-class Changes

# PyTorch
nn.CrossEntropyLoss()

# Last layer:
nn.Linear(64, n_classes)

# Predictions:
torch.argmax(logits, dim=1)

# TensorFlow
Dense(
    n_classes,
    activation="softmax"
)

model_tf.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)



# Common Losses

# Binary
nn.BCEWithLogitsLoss()
tf.keras.losses.BinaryCrossentropy()

# Multi-class
nn.CrossEntropyLoss()
tf.keras.losses.SparseCategoricalCrossentropy()



# Common Optimizers

optim.Adam(
    model_pt.parameters(),
    lr=0.001
)

optim.SGD(
    model_pt.parameters(),
    lr=0.01
)

tf.keras.optimizers.Adam()
tf.keras.optimizers.SGD()


# Save / Load

torch.save(
    model_pt.state_dict(),
    "model.pt"
)

model_tf.save(
    "model.keras"
)


# Notes
# - Scale features.
# - Binary: sigmoid + BCE loss.
# - Multi-class: softmax + cross-entropy.
# - Adam is a good default optimizer.
# - Use EarlyStopping to reduce overfitting.
# - Monitor validation accuracy/loss.

```